# States and values

Next to events, a lane can show what **state** it was in (up, degraded, down) and how a
**number** changed (CPU, latency). On the same lanes as the events they explain them.
Docs: <https://mluttikh.github.io/anytimeline/how-to/states-values.html>

In [ ]:
import numpy as np

from anytimeline import Timeline
from anytimeline.synthetic import service_health, service_log, service_metrics

services = ["gateway", "auth", "api", "cache", "db"]
log = service_log()
health = service_health()    # a row per change of state
metrics = service_metrics()  # a sample per service every 10 s

## Health, CPU and problems

States as the lanes' background (`height=1.0`, faint), the CPU as an area from 0 to 100%,
and the log's errors and warnings on top.

In [ ]:
problems = {k: v[log["level"] != "info"] for k, v in log.items()}

board = Timeline(title="Health, CPU and problems", height=420, lane_order=services)
board.states(health, lane="service", tooltip=["reason"],
             palette={"up": "#54a24b", "degraded": "#f58518", "down": "#e45756"},
             opacity=0.25, height=1.0)
board.values(metrics, value="cpu", lane="service", kind="area", ylim=(0, 100), name="cpu %")
board.events(problems, time="time", lane="service", color="level", symbol="event",
             palette={"error": "#c62828", "warning": "#ef6c00"}, name="problems")
board.zoom_to(np.datetime64("2026-10-12T08:25"), np.datetime64("2026-10-12T08:45"))
board

## Several lines per lane

`series=` splits a lane into several lines: the p50 and p99 latency of each service.
`kind="step"` holds each value until the next sample.

In [ ]:
latency = {
    "time": np.concatenate([metrics["time"], metrics["time"]]),
    "service": np.concatenate([metrics["service"], metrics["service"]]),
    "quantile": np.repeat(["p50", "p99"], len(metrics["time"])),
    "ms": np.concatenate([metrics["p50_ms"], metrics["p99_ms"]]),
}
Timeline(title="Latency per service", height=360, lane_order=services).values(
    latency, value="ms", lane="service", series="quantile", kind="step")